In [1]:
import torch
import torch.nn as nn 
import torch.nn.functional as F
import torchmetrics
import optuna

from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import random_split
from sklearn.datasets import fetch_covtype

/home/damian/test/test/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
class Dense(nn.Module):
    def __init__(self, hidden_1, hidden_2):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(hidden_2, hidden_1))
        nn.init.kaiming_uniform_(self.weight, nonlinearity='relu')
        self.bias = nn.Parameter(torch.zeros(hidden_2))
        
    def forward(self, X):
        z = X @ self.weight.T + self.bias
        return F.relu(z)



In [3]:
covertype = fetch_covtype()

X = torch.tensor(covertype.data, dtype=torch.float32)
X = (X - X.mean(dim=0, keepdim=True)) / X.std(dim=0, keepdim=True)

y = torch.tensor(covertype.target - 1, dtype=torch.long)

dataset = TensorDataset(X, y)

In [4]:
torch.manual_seed(52)

train_size = len(dataset) // 100 * 80
valid_size = len(dataset) // 100 * 10
test_size = len(dataset) - train_size - valid_size

train_data, valid_data, test_data = random_split(dataset, [train_size, valid_size, test_size])

In [5]:
batch_size = 256

train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
valid_dataloader = DataLoader(valid_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

In [10]:
def train_model(model, optimizer, criterion, data_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            
            optimizer.step()
            optimizer.zero_grad()
            
        mean_loss = total_loss / len(data_loader)
        if(epoch%2==0):
            print(f"epoch # {epoch}, loss: {mean_loss}")

In [11]:
def eval_model(model, data_loader, metrics):
    model.eval()
    metrics.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
            y_pred = model(X_batch)
            metrics.update(y_pred, y_batch)
            
    return metrics.compute()

In [16]:
model_1 = nn.Sequential(Dense(54, 200), Dense(200, 100), Dense(100, 50), nn.Linear(50, 7)).to('cuda')

for lr in [0.16, 0.08, 0.04, 0.02, 0.01, 0.005]:
    optimizer = torch.optim.SGD(params=model_1.parameters(), lr=lr)
    xentropy = nn.CrossEntropyLoss()
    
    train_model(model_1, optimizer, xentropy, train_dataloader, 15)

epoch # 0, loss: 0.5760558473709396
epoch # 2, loss: 0.404333587932823
epoch # 4, loss: 0.34418485509765046
epoch # 6, loss: 0.30874195541480826
epoch # 8, loss: 0.2826271992800388
epoch # 10, loss: 0.2642698082334407
epoch # 12, loss: 0.24957326601878924
epoch # 14, loss: 0.23797762430055552
epoch # 0, loss: 0.20000741038356584
epoch # 2, loss: 0.192225128248137
epoch # 4, loss: 0.1878738848221722
epoch # 6, loss: 0.18407236327651053
epoch # 8, loss: 0.1802768994467374
epoch # 10, loss: 0.17703215954291926
epoch # 12, loss: 0.18260703677319745
epoch # 14, loss: 0.17289471085540262
epoch # 0, loss: 0.14929518559209115
epoch # 2, loss: 0.14539347008888154
epoch # 4, loss: 0.1436076086990962
epoch # 6, loss: 0.14200997776208274
epoch # 8, loss: 0.1406057784497935
epoch # 10, loss: 0.1393016047923444
epoch # 12, loss: 0.1379506602054585
epoch # 14, loss: 0.13653825973427244
epoch # 0, loss: 0.12374539783076764
epoch # 2, loss: 0.1216975492199612
epoch # 4, loss: 0.12070396751378733
epoch 

In [18]:
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=7).to('cuda')
eval_model(model_1, valid_dataloader, accuracy)

tensor(0.9483, device='cuda:0')

In [19]:
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=7).to('cuda')
eval_model(model_1, test_dataloader, accuracy)

tensor(0.9496, device='cuda:0')